# 🍜 VNFood Vision — Training Pipeline trên Google Colab

**Hướng dẫn:**
1. Vào `Runtime` → `Change runtime type` → Chọn **T4 GPU** → Save
2. Chạy từng ô theo thứ tự từ trên xuống dưới
3. Chỉ cần sửa ô **CẤU HÌNH** (ô số 2) cho đúng đường dẫn Google Drive của bạn


## ⚙️ BƯỚC 0 — CẤU HÌNH (SỬA Ô NÀY TRƯỚC KHI CHẠY)

In [ ]:
# ============================================================
# 👇 CHỈ CẦN SỬA PHẦN NÀY
# ============================================================

# Đường dẫn đến thư mục dự án trên Google Drive của bạn
# (Thư mục chứa file README.md, Task.md, src/, configs/...)
PROJECT_DIR = "/content/drive/MyDrive/VietFood-Project/vnfood_vision"

# Đường dẫn đến thư mục ảnh đã xử lý
DATA_DIR = "/content/drive/MyDrive/VietFood-Project/data/processed"

# ============================================================
# KHÔNG CẦN SỬA BÊN DƯỚI
# ============================================================
import os
os.environ['VNFOOD_DATA_DIR'] = DATA_DIR
print(f"✅ Project dir : {PROJECT_DIR}")
print(f"✅ Data dir    : {DATA_DIR}")

## 📂 BƯỚC 1 — Mount Google Drive & Kiểm tra GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU
import torch
print(f"\n🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU — Vào Runtime > Change runtime type > T4 GPU'}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 📦 BƯỚC 2 — Cài đặt thư viện

In [ ]:
# Cài các thư viện cần thiết (Colab đã có torch/torchvision rồi)
!pip install -q pyyaml tqdm scikit-learn pillow
print("✅ Cài đặt xong!")

## 🗂️ BƯỚC 3 — Copy code vào Colab & Kiểm tra cấu trúc

In [ ]:
import shutil, os

# Copy toàn bộ thư mục dự án từ Drive vào /content/vnfood_vision
# (Đọc file từ /content/ nhanh hơn Drive ~10 lần)
LOCAL_PROJECT = "/content/vnfood_vision"

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(PROJECT_DIR, LOCAL_PROJECT)
os.chdir(LOCAL_PROJECT)
print(f"✅ Code đã copy vào: {LOCAL_PROJECT}")
print(f"📁 Thư mục hiện tại: {os.getcwd()}")

# Kiểm tra cấu trúc
!ls -la

In [ ]:
# Kiểm tra thư mục data (QUAN TRỌNG)
import os
from pathlib import Path

data_path = Path(DATA_DIR)
if data_path.exists():
    # Đếm số ảnh
    all_images = list(data_path.rglob('*.jpg')) + list(data_path.rglob('*.jpeg')) + list(data_path.rglob('*.png'))
    classes = [d.name for d in data_path.rglob('*') if d.is_dir() and not any(d.name.startswith(x) for x in ['.'])]
    print(f"✅ Tìm thấy thư mục data!")
    print(f"   Tổng số ảnh: {len(all_images):,}")
    print(f"   Số thư mục con: {len(classes)}")
    print(f"   Ví dụ 5 thư mục đầu: {[d.name for d in list(data_path.iterdir())[:5]]}")
else:
    print(f"❌ KHÔNG TÌM THẤY: {DATA_DIR}")
    print("   Kiểm tra lại đường dẫn DATA_DIR ở ô CẤU HÌNH")

## 🚀 BƯỚC 4 — TRAIN MÔ HÌNH

In [ ]:
# Chạy training!
# - Tự động dùng GPU T4
# - Tự động lưu model tốt nhất vào checkpoints/best_model.pth
# - Tự động dừng nếu không cải thiện sau 10 epoch (Early Stopping)

!python src/vision/train.py \
    --config configs/config.yaml \
    --backbone efficientnet_b3

In [ ]:
# [TÙY CHỌN] So sánh với backbone ResNet50
# Chạy ô này sau khi training EfficientNet xong để so sánh

# !python src/vision/train.py \
#     --config configs/config.yaml \
#     --backbone resnet50

## 🤖 BƯỚC 5 — ACTIVE LEARNING (AI tự lọc ảnh khả nghi)

In [ ]:
# Chạy Active Learning Scanner
# AI sẽ tự quét 27k ảnh và tự quyết định cần review bao nhiêu ảnh

!python src/vision/active_learning.py \
    --data_dir "{DATA_DIR}" \
    --model_path "checkpoints/best_model.pth"

In [ ]:
# Copy kết quả Active Learning về Google Drive để dùng với Label Studio
import shutil
from pathlib import Path

src = Path("/content/vnfood_vision/outputs/active_learning")
dst = Path(PROJECT_DIR) / "outputs" / "active_learning"
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    for f in src.iterdir():
        shutil.copy(f, dst / f.name)
    print(f"✅ Kết quả đã copy về Drive: {dst}")
    print(f"   File để import Label Studio: {dst / 'label_studio_import.json'}")
else:
    print("⚠️  Chưa có kết quả. Chạy ô Active Learning trước.")

## 🔍 BƯỚC 6 — TEST NHẬN DIỆN 1 ẢNH

In [ ]:
# Test nhận diện một ảnh bất kỳ
# Thay đường dẫn bên dưới bằng ảnh bạn muốn test

TEST_IMAGE = "/content/drive/MyDrive/VietFood-Project/test_anh.jpg"

!python src/vision/inference.py \
    --image "{TEST_IMAGE}" \
    --model_path "checkpoints/best_model.pth" \
    --tta  # Bật Test Time Augmentation để chính xác hơn

## 💾 BƯỚC 7 — LƯU MODEL VỀ GOOGLE DRIVE

In [ ]:
# Copy toàn bộ checkpoints về Drive để không mất khi Colab tắt
import shutil
from pathlib import Path

src = Path("/content/vnfood_vision/checkpoints")
dst = Path(PROJECT_DIR) / "checkpoints"
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    for f in src.iterdir():
        shutil.copy(f, dst / f.name)
        print(f"✅ Đã lưu: {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    print(f"\n📁 Model đã lưu tại Drive: {dst}")
else:
    print("⚠️  Chưa có checkpoint. Hãy chạy Training trước.")